In [1]:
!ls ../input/competitions/birdclef-2026

recording_location.txt	test_soundscapes  train_soundscapes
sample_submission.csv	train_audio	  train_soundscapes_labels.csv
taxonomy.csv		train.csv


In [2]:
BASE = "../input/competitions/birdclef-2026/"

In [3]:
import os
import ast
import torch
import torchaudio
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

TRAIN_AUDIO_DIR = os.path.join(BASE, "train_audio")
TRAIN_CSV = os.path.join(BASE, "train.csv")
TAXONOMY_CSV = os.path.join(BASE, "taxonomy.csv")

class Config:
    SR = 32000               # Sample rate specified by the competition
    DURATION = 5             # 5-second windows for prediction
    MAX_LENGTH = SR * DURATION # 160,000 samples
    N_MELS = 128             # Number of Mel frequency bins
    N_FFT = 1024
    HOP_LENGTH = 512
    BATCH_SIZE = 32
    NUM_WORKERS = 4

train_df = pd.read_csv(TRAIN_CSV)
taxonomy_df = pd.read_csv(TAXONOMY_CSV)

CLASSES = taxonomy_df['primary_label'].unique().tolist()
NUM_CLASSES = len(CLASSES)
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

train_df['target'] = train_df['primary_label'].map(class_to_idx)

In [4]:
class BirdCLEFDataset(Dataset):
    def __init__(self, df, audio_dir, config, is_train=True):
        self.df = df
        self.audio_dir = audio_dir
        self.config = config
        self.is_train = is_train
        
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=config.SR,
            n_fft=config.N_FFT,
            hop_length=config.HOP_LENGTH,
            n_mels=config.N_MELS,
            f_min=50,
            f_max=14000
        )
        
        self.amp_to_db = torchaudio.transforms.AmplitudeToDB()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        audio_path = os.path.join(self.audio_dir, row['filename'])
        
        waveform, sr = torchaudio.load(audio_path)
        
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        audio_len = waveform.shape[1]
        
        if audio_len > self.config.MAX_LENGTH:
            if self.is_train:
                max_start = audio_len - self.config.MAX_LENGTH
                start = np.random.randint(0, max_start)
            else:
                start = (audio_len - self.config.MAX_LENGTH) // 2
            waveform = waveform[:, start:start + self.config.MAX_LENGTH]
            
        elif audio_len < self.config.MAX_LENGTH:
            pad_len = self.config.MAX_LENGTH - audio_len
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))
            
        mel_spec = self.mel_transform(waveform)
        mel_spec = self.amp_to_db(mel_spec)
        
        mel_spec = (mel_spec - mel_spec.min()) / (mel_spec.max() - mel_spec.min() + 1e-6)
        mel_spec = mel_spec * 2 - 1
        
        mel_spec = mel_spec.expand(3, -1, -1)
        
        target = torch.zeros(NUM_CLASSES, dtype=torch.float32)
        
        primary_idx = class_to_idx.get(row['primary_label'])
        if primary_idx is not None:
            target[primary_idx] = 1.0
            
        secondary_labels = ast.literal_eval(row.get('secondary_labels', "[]"))
        for sec_label in secondary_labels:
            sec_idx = class_to_idx.get(sec_label)
            if sec_idx is not None:
                target[sec_idx] = 1.0

        return mel_spec, target

In [5]:
train_split, val_split = train_test_split(
    train_df, 
    test_size=0.2, 
    random_state=42 
)

train_dataset = BirdCLEFDataset(train_split, TRAIN_AUDIO_DIR, Config, is_train=True)
val_dataset = BirdCLEFDataset(val_split, TRAIN_AUDIO_DIR, Config, is_train=False)

train_loader = DataLoader(
    train_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=True, 
    num_workers=Config.NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=False, 
    num_workers=Config.NUM_WORKERS,
    pin_memory=True
)

In [6]:
import torch.nn as nn
import timm

class BirdCLEFModel(nn.Module):
    def __init__(self, model_name='efficientnet_b0', num_classes=234, pretrained=True):
        super().__init__()
        
        self.backbone = timm.create_model(
            model_name, 
            pretrained=pretrained, 
            num_classes=0 
        )
        
        in_features = self.backbone.num_features
        
        self.head = nn.Linear(in_features, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        
        logits = self.head(features)
        
        return logits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BirdCLEFModel().to(device)
print(f"Model loaded on {device}")

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Model loaded on cuda


In [7]:
import torch.optim as optim

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

model.train()

images, targets = next(iter(train_loader))
images = images.to(device)
targets = targets.to(device)

optimizer.zero_grad()
outputs = model(images)

loss = criterion(outputs, targets)

loss.backward()
optimizer.step()

print(f"Success! Forward and backward pass complete. Loss: {loss.item():.4f}")

Success! Forward and backward pass complete. Loss: 0.7006


In [8]:
import glob
import os
import torch
import torchaudio
import pandas as pd

TEST_AUDIO_DIR = os.path.join(BASE, "test_soundscapes")
test_files = glob.glob(os.path.join(TEST_AUDIO_DIR, "*.ogg"))

if len(test_files) == 0:
    print("Test directory empty. Using train_soundscapes for pipeline test...")
    TEST_AUDIO_DIR = os.path.join(BASE, "train_soundscapes")
    test_files = glob.glob(os.path.join(TEST_AUDIO_DIR, "*.ogg"))[:2] 

model.eval()
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=Config.SR, 
    n_fft=Config.N_FFT, 
    hop_length=Config.HOP_LENGTH, 
    n_mels=Config.N_MELS, 
    f_min=50, 
    f_max=14000
).to(device)
amp_to_db = torchaudio.transforms.AmplitudeToDB().to(device)

predictions = []

print(f"Processing {len(test_files)} files...")
with torch.no_grad():
    for file_path in test_files:
        filename = os.path.basename(file_path)
        file_id = filename.replace('.ogg', '')
        
        waveform, sr = torchaudio.load(file_path)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        waveform = waveform.to(device)
        
        chunk_length = Config.MAX_LENGTH # 160,000 samples
        num_chunks = waveform.shape[1] // chunk_length
        
        if num_chunks == 0:
            pad_len = chunk_length - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))
            num_chunks = 1
            
        for i in range(num_chunks):
            start = i * chunk_length
            end = start + chunk_length
            chunk = waveform[:, start:end]
            
            mel_spec = mel_transform(chunk)
            mel_spec = amp_to_db(mel_spec)
            mel_spec = (mel_spec - mel_spec.min()) / (mel_spec.max() - mel_spec.min() + 1e-6)
            mel_spec = mel_spec * 2 - 1
            
            mel_spec = mel_spec.unsqueeze(0).expand(-1, 3, -1, -1)
            
            logits = model(mel_spec)
            probs = torch.sigmoid(logits).cpu().numpy()[0]
            
            end_time = (i + 1) * 5 # e.g., 5, 10, 15... 60
            row_id = f"{file_id}_{end_time}"
            
            pred_dict = {'row_id': row_id}
            for class_name, prob in zip(CLASSES, probs):
                pred_dict[class_name] = prob
                
            predictions.append(pred_dict)

submission_df = pd.DataFrame(predictions)
submission_df.to_csv('submission.csv', index=False)
print("submission.csv successfully created!")

Test directory empty. Using train_soundscapes for pipeline test...
Processing 2 files...
submission.csv successfully created!


In [12]:
import time
import numpy as np
import torch
from sklearn.metrics import roc_auc_score
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

def calculate_competition_roc_auc(y_true, y_pred):
    aucs = []
    for i in range(y_true.shape[1]):
        if len(np.unique(y_true[:, i])) == 2:
            class_auc = roc_auc_score(y_true[:, i], y_pred[:, i])
            aucs.append(class_auc)
    if len(aucs) == 0:
        return 0.5
    return np.mean(aucs)

EPOCHS = 10
best_val_auc = 0.0

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

for epoch in range(EPOCHS):
    # Training
    model.train()
    train_loss = 0.0
    
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    
    for images, targets in train_pbar:
        images = images.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()
        
        logits = model(images)
        loss = criterion(logits, targets)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        
        train_pbar.set_postfix(batch_loss=f"{loss.item():.4f}")
        
    train_loss = train_loss / len(train_loader.dataset)
    
    # Validation
    model.eval()
    val_loss = 0.0
    
    all_val_targets = []
    all_val_preds = []
    
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]", leave=False)
    
    with torch.no_grad():
        for images, targets in val_pbar:
            images = images.to(device)
            targets = targets.to(device)
            
            logits = model(images)
            loss = criterion(logits, targets)
            val_loss += loss.item() * images.size(0)
            
            probs = torch.sigmoid(logits)
            
            all_val_targets.append(targets.cpu().numpy())
            all_val_preds.append(probs.cpu().numpy())
            
            val_pbar.set_postfix(batch_loss=f"{loss.item():.4f}")
            
    val_loss = val_loss / len(val_loader.dataset)
    
    all_val_targets = np.vstack(all_val_targets)
    all_val_preds = np.vstack(all_val_preds)
    
    val_auc = calculate_competition_roc_auc(all_val_targets, all_val_preds)
    
    # Logging
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()
    
    print(f"Epoch {epoch+1}/{EPOCHS} | LR: {current_lr:.2e}")
    print(f"  -> Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val ROC-AUC: {val_auc:.4f}")
    
    if val_auc > best_val_auc:
        print(f"  [+] Validation AUC improved ({best_val_auc:.4f} -> {val_auc:.4f}). Saving model!")
        best_val_auc = val_auc
        torch.save(model.state_dict(), 'best_birdclef_model.pth')
        
print("Training Complete!")

Starting Training...


Epoch 1/10 | LR: 1.00e-03
  -> Train Loss: 0.0235 | Val Loss: 0.0204 | Val ROC-AUC: 0.8993
  [+] Validation AUC improved (0.0000 -> 0.8993). Saving model!


Epoch 2/10 | LR: 9.76e-04
  -> Train Loss: 0.0189 | Val Loss: 0.0175 | Val ROC-AUC: 0.9246
  [+] Validation AUC improved (0.8993 -> 0.9246). Saving model!


Epoch 3/10 | LR: 9.05e-04
  -> Train Loss: 0.0164 | Val Loss: 0.0160 | Val ROC-AUC: 0.9372
  [+] Validation AUC improved (0.9246 -> 0.9372). Saving model!


Epoch 4/10 | LR: 7.94e-04
  -> Train Loss: 0.0149 | Val Loss: 0.0152 | Val ROC-AUC: 0.9412
  [+] Validation AUC improved (0.9372 -> 0.9412). Saving model!


...

Epoch 6/10 | LR: 5.01e-04
  -> Train Loss: 0.0124 | Val Loss: 0.0140 | Val ROC-AUC: 0.9509
  [+] Validation AUC improved (0.9473 -> 0.9509). Saving model!


Epoch 7/10 | LR: 3.46e-04
  -> Train Loss: 0.0115 | Val Loss: 0.0134 | Val ROC-AUC: 0.9542
  [+] Validation AUC improved (0.9509 -> 0.9542). Saving model!


Epoch 8/10 | LR: 2.07e-04
  -> Train Loss: 0.0106 | Val Loss: 0.0131 | Val ROC-AUC: 0.9545
  [+] Validation AUC improved (0.9542 -> 0.9545). Saving model!


Epoch 9/10 | LR: 9.64e-05
  -> Train Loss: 0.0099 | Val Loss: 0.0130 | Val ROC-AUC: 0.9559
  [+] Validation AUC improved (0.9545 -> 0.9559). Saving model!


Epoch 10/10 | LR: 2.54e-05
  -> Train Loss: 0.0095 | Val Loss: 0.0130 | Val ROC-AUC: 0.9560
  [+] Validation AUC improved (0.9559 -> 0.9560). Saving model!
Training Complete!


**Public Leaderboard Score: 0.781**

In [14]:
model.load_state_dict(torch.load('best_birdclef_model.pth'))
model.to(device)
model.eval()

BirdCLEFModel(
  (backbone): EfficientNet(
    (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
          )
          (conv_pw):